# 22 - Simulation And Domain Randomization

## What / Why / How

**What we are trying to do:** Train and evaluate a controller across randomized simulated physics.

**Why this matters:** Simulators are useful but imperfect. Robust policies must survive variation in mass, friction, sensors, and actuators.

**How we will do it:** Randomize dynamics parameters, sweep controller gains, choose a robust setting, and compare nominal versus randomized performance.

## Goal

Simulation is essential, but simulators are wrong in small ways.

You will use domain randomization to train a controller that survives variation in:

- Mass
- Friction
- Actuator scale
- Sensor noise

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(22)

def run_episode(gain, randomized=True):
    mass = rng.uniform(0.6, 1.8) if randomized else 1.0
    actuator = rng.uniform(0.75, 1.25) if randomized else 1.0
    friction = rng.uniform(0.02, 0.2) if randomized else 0.08
    x, v = -1.0, 0.0
    target = 1.0
    total_error = 0.0
    for _ in range(150):
        observed_x = x + rng.normal(0, 0.02 if randomized else 0.0)
        force = actuator * np.clip(gain * (target - observed_x) - 1.5*v, -4, 4)
        a = force / mass - friction * v
        v += a * 0.02
        x += v * 0.02
        total_error += abs(target - x)
    return total_error / 150, abs(target - x)

gains = np.linspace(0.5, 12.0, 30)
scores = []
for gain in gains:
    errors = [run_episode(gain, randomized=True)[1] for _ in range(60)]
    scores.append(np.mean(errors))
best_gain = float(gains[int(np.argmin(scores))])
print("best randomized gain:", best_gain)

In [ ]:
nominal_errors = [run_episode(best_gain, randomized=False)[1] for _ in range(20)]
randomized_errors = [run_episode(best_gain, randomized=True)[1] for _ in range(100)]
print("nominal final error mean:", np.mean(nominal_errors))
print("randomized final error mean:", np.mean(randomized_errors))
print("randomized 90th percentile:", np.percentile(randomized_errors, 90))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(7, 3))
    plt.plot(gains, scores)
    plt.axvline(best_gain, color="tab:red", linestyle="--")
    plt.xlabel("gain")
    plt.ylabel("mean final error")
    plt.grid(True, alpha=0.3)
    plt.title("Domain randomization sweep")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add latency to the simulator.
2. Train on nominal physics and test on randomized physics.
3. Explain why domain randomization is not a substitute for real tests.